In [55]:
import openai, json, requests

client = openai.OpenAI()
messages = []

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies():
    url = BASE_URL + "/movies"
    r = requests.get(url)

    movies = []
    for data in json.loads(r.text):
        movie = {
            "title": data["title"],
            "id": data["id"]
        }
        movies.append(movie)

    return json.dumps(movies)

def get_movie_details(id: int):
    url = BASE_URL + f"/movies/{id}"
    r = requests.get(url)
    return r.text

def get_similar_movies(id: int):
    url = BASE_URL + f"/movies/{id}/similar"
    r = requests.get(url)
    return r.text

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "get popurlar movies list",
            "parameters": {
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Providing detailed information about the requested movie",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "number",
                        "description": "The id of the movie to get the details.",
                    }
                },
                "required": ["id"],
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Recommendations for movies similar to the requested movie",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "number",
                        "description": "recommend a movie similar to this id's movie",
                    }
                },
                "required": ["id"],
            },
        }
    }
]

In [ ]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            arguments = json.loads(arguments)

            function_to_run = FUNCTION_MAP.get(function_name)
            result = function_to_run(**arguments)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )
            print(f"Agent: [{function_name}("+', '.join([str(x) for x in list(arguments.values())])+") 호출]")
        
        call_ai()
    else:
            messages.append({"role": "assistant", "content": message.content})
            print(f"Agent: {message.content}")

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)


In [57]:
while True:
    message = input("User: ")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()

User: 요즘 인기 있는 영화 알려줘
Agent: [get_popular_movies() 호출]
Agent: 요즘 인기 있는 영화 리스트입니다:

1. **Your Heart Will Be Broken** (ID: 1523145)
2. **Avatar: Fire and Ash** (ID: 83533)
3. **Hoppers** (ID: 1327819)
4. **The Super Mario Bros. Movie** (ID: 502356)
5. **Shelter** (ID: 1290821)
6. **Crime 101** (ID: 1171145)
7. **Greenland 2: Migration** (ID: 840464)
8. **Project Hail Mary** (ID: 687163)
9. **GOAT** (ID: 1297842)
10. **The Super Mario Galaxy Movie** (ID: 1226863)
11. **Mike & Nick & Nick & Alice** (ID: 1115544)
12. **Sniper: No Nation** (ID: 1641319)
13. **The Bride!** (ID: 1159831)
14. **War Machine** (ID: 1265609)
15. **The Mortuary Assistant** (ID: 1470130)
16. **Pretty Lethal** (ID: 1084187)
17. **Humint** (ID: 1268127)
18. **Scream 7** (ID: 1159559)
19. **The Passion of the Christ** (ID: 615)
20. **Starbright** (ID: 662707)

이 중에서 더 알고 싶은 영화가 있으면 말씀해 주세요!
User: Your Heart Will Be Broken 에 대해 더 알려줘


TypeError: sequence item 0: expected str instance, int found